# Chapter 4 — Transfer Learning and Fine-Tuning

Chapters 2 and 3 trained every weight from scratch, and both ran into the same
wall: results scale with how much labelled data you have, and you rarely have
enough. This chapter is the escape hatch, and it is the single most useful
practical technique in the book. Almost nobody outside a frontier lab trains a
vision model from a random initialization.

The idea is that the early layers of any image CNN learn roughly the *same*
things: edges, textures, colour blobs, exactly what you looked at in Chapter 2,
Module 4. So borrow them from somebody who could afford to train on a million
photos, and spend your own data only on what is specific to your problem.

There are two versions of that, and the gap between them is the chapter:

| Module | What you build | Dataset (Hugging Face) |
|---|---|---|
| 1 | **Frozen backbone**: cache the embeddings, train a tiny head on top | `AI-Lab-Makerere/beans` |
| 2 | **Fine-tuning**: unfreeze the top of the backbone and specialize it, carefully | `AI-Lab-Makerere/beans` |

Both work on the same task, so the numbers are directly comparable: diagnosing
**bean plant leaves** from Uganda (healthy vs. two diseases), with about
**1,000 training photos**. Hopeless from scratch; easy with transfer.

Module 2 is where the sharp edges live. Fine-tuning done naively destroys the
very thing you borrowed. The two standard ways to get it wrong are updating too
early and letting batch normalization drift, and the recipe explains what each
line is guarding against rather than leaving you a warning to take on trust.

**How each concept is presented**, the same three passes as Chapters 1–3:

> 🧠 **The intuition:** the idea in plain language, no symbols.
> 📐 **The math:** the same idea written precisely, so you can read papers.
> 💻 **The code:** the same idea again, executable, in the cell that follows.

**Runtime:** ~15 min on CPU, most of it in Module 2, since every epoch there
pushes all 1,034 images through the full backbone, which is exactly the cost
Module 1 avoids. The beans dataset (~170 MB) downloads once and is then cached.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from datasets import load_dataset

keras.utils.set_random_seed(42)
rng = np.random.default_rng(seed=42)
plt.rcParams["figure.figsize"] = (7, 4.5)

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

---
# Module 1 — Borrowed Features: The Frozen Backbone

🧠 **The intuition.** Take **MobileNetV2** pretrained on ImageNet (1.4M photos,
1000 classes). Chop off its classification head. What is left is a **backbone**
that turns any photo into a 1280-number description: "how much edge, how much
green, how much fur-like texture, how much wheel-like curve". That description
was learned from a million photos, and it is a *generic* description of images,
not an ImageNet-specific one.

So: freeze it, run every bean leaf through it once, and train a three-class
classifier on the resulting 1280-number vectors. You are training a logistic
regression on somebody else's features.

📐 **What this actually costs.** Because the backbone is frozen, each image's
embedding is a *constant* and never changes during training. So you compute it
**once**, cache it, and every subsequent epoch touches only the tiny head. Each
epoch drops from minutes to milliseconds, and the division of labour is explicit:

$$ \underbrace{E_{\text{ImageNet}}(x)}_{\text{2.2M frozen parameters}} \;\longrightarrow\; \underbrace{h_\theta(\cdot)}_{\text{\textasciitilde 3{,}800 trained parameters}} $$

We train **0.2%** of the network.

💻 **The code.**

## 1.1 Load and look


In [ ]:
beans = load_dataset("AI-Lab-Makerere/beans")     # ~170 MB on first run
print(beans)

bean_classes = beans["train"].features["labels"].names   # note: column is "labels" here
print(bean_classes)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
shuffled = beans["train"].shuffle(seed=1)
for cls, ax in enumerate(axes):
    row = next(r for r in shuffled if r["labels"] == cls)
    ax.imshow(row["image"])
    ax.set_title(bean_classes[cls], fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 1.2 Resize and preprocess

The photos are 500×500; MobileNetV2 is happy at 160×160. Every pretrained model
has a matching `preprocess_input` that reproduces the exact normalization used
during its original training. Skipping it is the classic transfer-learning bug,
and it fails silently: the model still trains, just worse, with no error to read.


In [ ]:
IMG = 160

def beans_to_arrays(split):
    images = np.stack([np.array(im.resize((IMG, IMG))) for im in split["image"]])
    labels = np.array(split["labels"])
    return images, labels

Xb_train, yb_train = beans_to_arrays(beans["train"])        # 1034 images
Xb_val,   yb_val   = beans_to_arrays(beans["validation"])   # 133
Xb_test,  yb_test  = beans_to_arrays(beans["test"])         # 128
print("train:", Xb_train.shape, "| val:", Xb_val.shape, "| test:", Xb_test.shape)

## 1.3 Extract features with the frozen backbone

💻 `pooling="avg"` averages the final 5×5 feature maps into one **1280-number
embedding** per image, which is MobileNetV2's learned summary of everything it
saw. We compute all of them up front, because a frozen backbone produces the
same embedding every epoch and recomputing it would be pure waste.


In [ ]:
backbone = keras.applications.MobileNetV2(
    input_shape=(IMG, IMG, 3),
    include_top=False,           # drop the 1000-class ImageNet head
    weights="imagenet",          # ~9 MB download on first run
    pooling="avg",               # (5, 5, 1280) feature maps -> (1280,) embedding
)
backbone.trainable = False       # freeze: 2.2M pretrained weights stay untouched
preprocess = keras.applications.mobilenet_v2.preprocess_input   # scales pixels to [-1, 1]

E_train = backbone.predict(preprocess(Xb_train.astype("float32")), batch_size=32, verbose=1)
E_val   = backbone.predict(preprocess(Xb_val.astype("float32")),   batch_size=32, verbose=0)
E_test  = backbone.predict(preprocess(Xb_test.astype("float32")),  batch_size=32, verbose=0)
print("embeddings:", E_train.shape)

## 1.4 Train the head: a few seconds, not hours

The entire trainable model is one dropout + one dense layer: ~3,800 parameters
against MobileNetV2's 2.2 million frozen ones.


In [ ]:
head = keras.Sequential([
    layers.Input(shape=(E_train.shape[1],)),   # the 1280-dim embedding
    layers.Dropout(0.3),
    layers.Dense(3, activation="softmax"),     # healthy / angular_leaf_spot / bean_rust
], name="beans_head")

head.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist = head.fit(E_train, yb_train,
                validation_data=(E_val, yb_val),
                epochs=30, batch_size=32, verbose=0)   # seconds, because the embeddings are precomputed

loss, acc = head.evaluate(E_test, yb_test, verbose=0)
print(f"beans test accuracy with ~1000 training images: {acc:.4f}")

plt.plot(hist.history["accuracy"], label="train")
plt.plot(hist.history["val_accuracy"], label="validation")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.grid(True)
plt.title("Transfer-learning head on beans"); plt.legend()
plt.show()

Around **90%+ accuracy from ~1,000 images**, because 2.2M of the "learned"
parameters arrived pretrained. Compare that with Chapter 2, Module 3, where
10,000 images bought us ~65–70% from scratch.

**Write this number down.** Module 2 attacks the same task with the same
backbone and the restriction lifted, and the whole point of the chapter is the
size of the gap.

## 1.5 Sanity check: predictions on real leaves


In [ ]:
probs_b = head.predict(E_test, verbose=0)
preds_b = probs_b.argmax(axis=1)

pick = rng.choice(len(Xb_test), size=6, replace=False)
fig, axes = plt.subplots(1, 6, figsize=(13, 2.9))
for ax, idx in zip(axes, pick):
    ax.imshow(Xb_test[idx])
    ok = preds_b[idx] == yb_test[idx]
    ax.set_title(f"{bean_classes[preds_b[idx]]}\n{probs_b[idx].max():.0%} {'✓' if ok else '✗ (' + bean_classes[yb_test[idx]] + ')'}",
                 fontsize=8, color="green" if ok else "red")
    ax.axis("off")
plt.tight_layout()
plt.show()

---
# Module 2 — Fine-Tuning: From Borrowed Features to Specialized Ones

🧠 **The intuition.** Module 1's frozen backbone describes leaves in
*ImageNet's* terms: generic textures, colors, shapes. Nothing in it has ever
seen leaf rust. **Fine-tuning** lets the top of the backbone *specialize* to our
domain, and the reason it needs a recipe rather than a flag is that doing it
naively destroys the very thing you borrowed.

📐 **The professional two-stage recipe:**

1. **Stage A, train the head with the backbone frozen.** The
   randomly-initialized head produces garbage gradients at first; if the
   backbone were unfrozen now, those gradients would wreck its pretrained
   weights.
2. **Stage B, unfreeze the top of the backbone, drop the learning rate ~100×,
   and keep training.** Small, careful updates specialize the high-level
   features without erasing them (avoiding *catastrophic forgetting*).

Note which end gets unfrozen. Early layers detect edges and textures, which are
the same for ImageNet and for bean leaves; late layers detect ImageNet-specific
compositions, which are not. So the top specializes and the bottom stays put.

⚠️ **One sharp edge: batch normalization.** The backbone's BN layers carry
running statistics accumulated over ImageNet. If they start updating on our
small dataset mid-training, the features shift under the head's feet and
accuracy craters. The fix is calling the backbone with `training=False`
*permanently*, so BN always runs in inference mode, even during Stage B when
its neighboring conv weights are learning.

This `training=False` is a **different switch** from `trainable`, which controls
whether weights receive gradient updates. Confusing the two is the classic
fine-tuning bug, and it produces a model that trains smoothly and evaluates
terribly.

## 2.1 Build the full end-to-end model

💻 Unlike Module 1, where we precomputed embeddings (fast, but impossible to
fine-tune *through*), the backbone now lives *inside* the model so gradients can
reach it in Stage B. Augmentation lives inside too, and
`Rescaling(1/127.5, -1)` reproduces MobileNetV2's required $[-1, 1]$
preprocessing as a layer.

(The bean arrays from Module 1 are still in memory, so there is nothing to
reload.)


In [ ]:
backbone = keras.applications.MobileNetV2(
    input_shape=(IMG, IMG, 3),
    include_top=False,
    weights="imagenet",
    pooling="avg",                                   # -> one 1280-dim embedding per image
)
backbone.trainable = False                           # Stage A: fully frozen

inputs = keras.Input(shape=(IMG, IMG, 3))
x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomRotation(0.1)(x)
x = layers.Rescaling(1 / 127.5, offset=-1)(x)        # MobileNetV2 expects pixels in [-1, 1]
x = backbone(x, training=False)                      # BN in inference mode FOREVER (see above)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(bean_classes), activation="softmax")(x)

ft_model = keras.Model(inputs, outputs, name="beans_finetune")
ft_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss="sparse_categorical_crossentropy",
                 metrics=["accuracy"])
print(f"trainable params (Stage A): {sum(int(np.prod(w.shape)) for w in ft_model.trainable_weights):,}")

## 2.2 Stage A: train the head (backbone frozen)

Every epoch now pushes all 1,034 images through the full backbone, so this is
far slower per-epoch than Module 1's cached-embedding trick. That is the price
of keeping the door open for Stage B, and it is worth knowing you are paying
it, because if you never intend to unfreeze, Module 1's approach is strictly
better.


In [ ]:
Xb_train_f = Xb_train.astype("float32")   # Rescaling layer handles normalization from raw pixels
Xb_val_f   = Xb_val.astype("float32")
Xb_test_f  = Xb_test.astype("float32")

EPOCHS_A = 4
hist_a = ft_model.fit(Xb_train_f, yb_train,
                      validation_data=(Xb_val_f, yb_val),
                      epochs=EPOCHS_A, batch_size=32, verbose=2)

loss_a, acc_frozen = ft_model.evaluate(Xb_test_f, yb_test, verbose=0)
print(f"Stage A (frozen backbone) test accuracy: {acc_frozen:.4f}")

## 2.3 Stage B: unfreeze the top, drop the learning rate 100×

We unfreeze roughly the last fifth of the backbone (the most task-specific
layers; early edge/texture layers stay frozen) and recompile with `1e-5`.
**Recompiling is mandatory**, because `trainable` changes only take effect at
compile, so a forgotten `compile()` here silently trains nothing new.


In [ ]:
backbone.trainable = True
for layer in backbone.layers[:-30]:      # freeze everything except the last ~30 layers
    layer.trainable = False

ft_model.compile(optimizer=keras.optimizers.Adam(1e-5),   # ~100x smaller than Stage A
                 loss="sparse_categorical_crossentropy",
                 metrics=["accuracy"])
print(f"trainable params (Stage B): {sum(int(np.prod(w.shape)) for w in ft_model.trainable_weights):,}")

EPOCHS_B = 4
hist_b = ft_model.fit(Xb_train_f, yb_train,
                      validation_data=(Xb_val_f, yb_val),
                      epochs=EPOCHS_A + EPOCHS_B, initial_epoch=EPOCHS_A,
                      batch_size=32, verbose=2)

loss_b, acc_finetuned = ft_model.evaluate(Xb_test_f, yb_test, verbose=0)
print(f"Stage B (fine-tuned) test accuracy: {acc_finetuned:.4f}  (was {acc_frozen:.4f} frozen)")

In [ ]:
val_acc = hist_a.history["val_accuracy"] + hist_b.history["val_accuracy"]
tr_acc  = hist_a.history["accuracy"]     + hist_b.history["accuracy"]

plt.plot(tr_acc, marker="o", label="train")
plt.plot(val_acc, marker="o", label="validation")
plt.axvline(EPOCHS_A - 0.5, color="gray", linestyle="--")
plt.text(EPOCHS_A - 0.4, min(tr_acc) + 0.01, "unfreeze + LR drop", rotation=90,
         va="bottom", fontsize=9, color="gray")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.grid(True); plt.legend()
plt.title("Two-stage fine-tuning on beans")
plt.show()

## 2.4 Look at what changed

Aggregate accuracy is up, but *which* leaves did fine-tuning rescue? Compare a
few test predictions with their confidence.


In [ ]:
probs = ft_model.predict(Xb_test_f, verbose=0)
preds = probs.argmax(axis=1)

pick = rng.choice(len(Xb_test), size=6, replace=False)
fig, axes = plt.subplots(1, 6, figsize=(13, 2.9))
for ax, idx in zip(axes, pick):
    ax.imshow(Xb_test[idx])
    ok = preds[idx] == yb_test[idx]
    title = f"{bean_classes[preds[idx]]}\n{probs[idx].max():.0%}"
    if not ok:
        title += f" (true: {bean_classes[yb_test[idx]]})"
    ax.set_title(title, fontsize=8, color="green" if ok else "red")
    ax.axis("off")
plt.tight_layout()
plt.show()

wrong = np.flatnonzero(preds != yb_test)
print(f"{len(wrong)} errors on {len(yb_test)} test leaves ({1 - len(wrong)/len(yb_test):.1%} accuracy)")

---
# Wrap-Up

| You built | The transferable lesson |
|---|---|
| Frozen backbone + cached embeddings | A frozen backbone makes each embedding a constant, so compute it once and every epoch is milliseconds |
| A 3,800-parameter head on 2.2M frozen weights | ~90% from 1,000 photos, because most of the "learning" arrived pretrained |
| Two-stage fine-tuning | Head first at high LR, then top-of-backbone at ~100× lower; unfreeze the *late* layers, never the early ones |
| `training=False` vs `trainable` | Two different switches. Keep the backbone's BN in inference mode or its running statistics drift and accuracy craters |

**When to use which.** Frozen backbone if your data is tiny, your domain is
close to the pretraining domain, or you need to iterate fast, since it is cheap
and nearly impossible to get wrong. Fine-tuning when you have enough data to
support it and your domain is far enough from ImageNet that generic features
leave accuracy on the table.

**Where this idea goes next.** Everything in this chapter is transfer learning
for *convolutional vision* models, but the pattern generalizes without change:
pretrain something enormous on a task where labels are free, then adapt it
cheaply to the task you actually care about. The generative models in the next
chapters are pretraining objectives of exactly that kind, and they need no
labels at all.
